# 🌿 Quality of Life in São Paulo State — Find Your Town

What makes a place truly great to live in? A thriving economy? Green space and clean air? A mild climate? Good schools and long, healthy lives?

Quality of life is personal — but data can help narrow it down.

This tool scores all **645 municipalities** in São Paulo State across indicators that research consistently links to wellbeing, then lets you adjust the weights to match *your* definition of a good life.

---

| Indicator | Source | What it captures |
|---|---|---|
| Human Development Index (IDHM) | IPEADATA | Longevity, education, income |
| GDP per capita | IBGE SIDRA | Local economic opportunity |
| Forest & vegetation coverage | MapBiomas Collection 10 | Green space, air quality |
| Climate (temp, rain, humidity, sunshine) | INMET + Open-Meteo | Comfort and liveability |
| Population density | IBGE SIDRA + Census 2022 | Urban pressure vs. peace |


---
## Setup


In [ ]:
# Install required packages
!pip install pandas numpy matplotlib seaborn requests plotly geopandas folium scikit-learn openpyxl ipywidgets -q


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import json
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('✅ Libraries loaded')

---
## Data Pipeline

All data is fetched automatically from public APIs. Run this section once — results are stored in `df`.


### 1. Municipality List — IBGE


In [ ]:
# Fetch all municipalities in São Paulo (UF code = 35)
url_mun = 'https://servicodados.ibge.gov.br/api/v1/localidades/estados/35/municipios'
resp = requests.get(url_mun)
municipios_raw = resp.json()

municipios = pd.DataFrame([{
    'cod_ibge': m['id'],
    'nome': m['nome'],
    'microrregiao': m['microrregiao']['nome'],
    'mesorregiao': m['microrregiao']['mesorregiao']['nome']
} for m in municipios_raw])

municipios['cod_ibge'] = municipios['cod_ibge'].astype(int)

print(f'Total municipalities in SP: {len(municipios)}')
municipios.head(10)

### 2. Population — IBGE Census 2022


In [ ]:
# --- 2a. Population Census 2022 (Table 9514) ---
# n6/all = all municipalities; we filter SP afterwards
# v/93 = resident population; p/last = most recent period
url_pop = (
    'https://apisidra.ibge.gov.br/values'
    '/t/9514/n6/all/v/93/p/last'
    '/d/v93%200'
)
resp_pop = requests.get(url_pop)
pop_raw = resp_pop.json()

# First row is header metadata
pop_df = pd.DataFrame(pop_raw[1:])
pop_df = pop_df.rename(columns={
    'D1C': 'cod_ibge',
    'D1N': 'nome_municipio',
    'V': 'populacao_2022'
})
pop_df['cod_ibge'] = pop_df['cod_ibge'].astype(int)
pop_df['populacao_2022'] = pd.to_numeric(pop_df['populacao_2022'], errors='coerce')

# Filter to SP only (codes starting with 35)
pop_sp = pop_df[pop_df['cod_ibge'].astype(str).str.startswith('35')][['cod_ibge', 'populacao_2022']].copy()
pop_sp = pop_sp.reset_index(drop=True)

print(f'SP municipalities with population data: {len(pop_sp)}')
pop_sp.sort_values('populacao_2022', ascending=False).head(10)

In [ ]:
# Merge population into main dataframe
df = municipios.merge(pop_sp, on='cod_ibge', how='left')
print(f'Records after merge: {len(df)}')
df.sort_values('populacao_2022', ascending=False).head(10)

### 3. GDP per Capita — IBGE SIDRA


In [ ]:
# GDP per capita — Table 5938, variable 513
url_gdp = (
    'https://apisidra.ibge.gov.br/values'
    '/t/5938/n6/all/v/513,37/p/last'
)
resp_gdp = requests.get(url_gdp)
gdp_raw = resp_gdp.json()

gdp_df = pd.DataFrame(gdp_raw[1:])
gdp_df['cod_ibge'] = gdp_df['D1C'].astype(int)
gdp_df['variable'] = gdp_df['D2N']
gdp_df['valor'] = pd.to_numeric(gdp_df['V'], errors='coerce')
gdp_df['ano'] = gdp_df['D3N']

# Filter SP
gdp_sp = gdp_df[gdp_df['cod_ibge'].astype(str).str.startswith('35')].copy()

# Pivot: one row per municipality
gdp_pivot = gdp_sp.pivot_table(index='cod_ibge', columns='D2C', values='valor', aggfunc='first').reset_index()
gdp_pivot.columns = ['cod_ibge'] + ['pib_mil_reais' if col == '37' else 'pib_per_capita' if col == '513' else col for col in gdp_pivot.columns[1:]]

print(f'Reference year: {gdp_sp["ano"].iloc[0]}')
print(f'SP municipalities with GDP data: {len(gdp_pivot)}')

df = df.merge(gdp_pivot, on='cod_ibge', how='left')
df.sort_values('pib_mil_reais', ascending=False).head(15)[['nome', 'populacao_2022', 'pib_mil_reais']]

### 4. Human Development Index — IPEADATA


In [ ]:
# Fetch IDHM 2010 for all municipalities via IPEADATA OData API
# Series ADH_IDHM: composite Municipal HDI (1991, 2000, 2010)
IPEADATA_URL = "http://ipeadata.gov.br/api/odata4/ValoresSerie(SERCODIGO='ADH_IDHM')"

try:
    all_records = []
    url = f"{IPEADATA_URL}?$select=VALDATA,NIVNOME,TERCODIGO,VALVALOR"
    while url:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        all_records.extend(data['value'])
        url = data.get('@odata.nextLink')

    idhm_raw = pd.DataFrame(all_records)

    # Keep only 2010 data for SP municipalities (7-digit codes starting with '35')
    idhm_sp = idhm_raw[
        (idhm_raw['TERCODIGO'].str.startswith('35')) &
        (idhm_raw['VALDATA'] == '2010-01-01T00:00:00-02:00') &
        (idhm_raw['TERCODIGO'].str.len() == 7)   # municipalities only
    ][['TERCODIGO', 'VALVALOR']].copy()

    idhm_sp.columns = ['cod_ibge_str', 'idhm']
    idhm_sp['cod_ibge'] = idhm_sp['cod_ibge_str'].astype(int)

    print(f'SP municipalities with IDHM 2010: {len(idhm_sp)}')
    idhm_sp.sort_values('idhm', ascending=False).head(10)
except Exception as e:
    print(f'IPEADATA fetch failed: {e}')
    idhm_sp = None


In [ ]:
# Merge IDHM into main dataframe
if idhm_sp is not None:
    df = df.merge(idhm_sp[['cod_ibge', 'idhm']], on='cod_ibge', how='left')
    print(f'IDHM merged — coverage: {df["idhm"].notna().sum()}/{len(df)} municipalities')
    print(df[['nome', 'populacao_2022', 'idhm']].sort_values('idhm', ascending=False).head(10).to_string(index=False))
else:
    df['idhm'] = float('nan')
    print('IDHM not available — column added as NaN')
    print("""
Expected IDHM columns after download:
  - cod_ibge (7-digit code)
  - idhm         (composite 0-1)

Fallback download: https://www.atlasbrasil.org.br/acervo/
""")


### 5. Forest Coverage — MapBiomas Collection 10

Downloads ~75 MB on first run, then uses the cached file.


In [ ]:
import os

MAPBIOMAS_FILE = 'mapbiomas_col10_municipalities.xlsx'
MAPBIOMAS_URL  = 'https://data.mapbiomas.org/api/v1/access/datafile/254'  # Collection 10

# Download once and cache locally (~75 MB)
if not os.path.exists(MAPBIOMAS_FILE):
    print('Downloading MapBiomas Collection 10 (~75 MB) — one-time download...')
    resp_mb = requests.get(MAPBIOMAS_URL, timeout=180, stream=True)
    resp_mb.raise_for_status()
    with open(MAPBIOMAS_FILE, 'wb') as fh:
        for chunk in resp_mb.iter_content(chunk_size=1024 * 1024):
            fh.write(chunk)
    print(f'Saved {os.path.getsize(MAPBIOMAS_FILE)/1024/1024:.0f} MB')
else:
    print(f'Using cached file ({os.path.getsize(MAPBIOMAS_FILE)/1024/1024:.0f} MB)')

# Load and filter São Paulo
mb_raw = pd.read_excel(MAPBIOMAS_FILE, sheet_name='COVERAGE_10')
mb_sp  = mb_raw[mb_raw['state'].str.contains('Paulo', na=False)].copy()
print(f'SP rows: {len(mb_sp)}, municipalities: {mb_sp["geocode"].nunique()}')
mb_sp.head(3)


In [ ]:
# Natural forest classes per MapBiomas legend:
#   3  = Forest Formation
#   4  = Savanna Formation (Cerrado)
#   5  = Mangrove
#   49 = Wooded Sandbank Vegetation
FOREST_CLASSES = [3, 4, 5, 49]

total_area  = mb_sp.groupby('geocode')[2024].sum().rename('total_area_ha')
forest_area = mb_sp[mb_sp['class'].isin(FOREST_CLASSES)].groupby('geocode')[2024].sum().rename('forest_area_ha')

tree_cov = pd.concat([total_area, forest_area], axis=1).fillna(0)
tree_cov['forest_pct'] = (tree_cov['forest_area_ha'] / tree_cov['total_area_ha'] * 100).round(2)
tree_cov = tree_cov.reset_index().rename(columns={'geocode': 'cod_ibge'})

df = df.merge(tree_cov[['cod_ibge', 'forest_pct']], on='cod_ibge', how='left')
print(f'forest_pct merged — coverage: {df["forest_pct"].notna().sum()}/{len(df)} municipalities')
print('\nTop 15 greenest municipalities:')
df.sort_values('forest_pct', ascending=False).head(15)[['nome', 'mesorregiao', 'populacao_2022', 'forest_pct']]


### 6. Climate — INMET Stations + Open-Meteo (2019–2023)

Fetches 5-year climate averages per station, then assigns each municipality its nearest station via Haversine distance.


In [ ]:
# Step 1: Fetch INMET automatic station list for SP
try:
    r_st = requests.get('https://apitempo.inmet.gov.br/estacoes/T', timeout=15)
    r_st.raise_for_status()
    stations = pd.DataFrame(r_st.json())
    stations_sp = stations[stations['SG_ESTADO'] == 'SP'].copy()
    stations_sp['lat'] = pd.to_numeric(stations_sp['VL_LATITUDE'])
    stations_sp['lon'] = pd.to_numeric(stations_sp['VL_LONGITUDE'])
    print(f'INMET automatic stations in SP: {len(stations_sp)}')
    display(stations_sp[['DC_NOME', 'CD_ESTACAO', 'lat', 'lon']].head(10))
except Exception as e:
    print(f'INMET station list error: {e}')
    stations_sp = None

# Step 2: Fetch 5-year climate history via Open-Meteo archive API (free, no auth)
# Variables fetched:
#   temperature_2m_mean   -> temp_mean_c, comfortable_days_yr (18–26°C)
#   temperature_2m_max    -> hot_days_yr (>35°C)
#   precipitation_sum     -> precip_annual_mm
#   relative_humidity_2m_mean -> humidity_mean_pct
#   sunshine_duration     -> sunshine_hrs_yr (seconds/day -> hours/yr)
if stations_sp is not None:
    params_om = {
        'latitude':   stations_sp['lat'].tolist(),
        'longitude':  stations_sp['lon'].tolist(),
        'start_date': '2019-01-01',
        'end_date':   '2023-12-31',
        'daily':      'temperature_2m_mean,temperature_2m_max,precipitation_sum,relative_humidity_2m_mean,sunshine_duration',
        'timezone':   'America/Sao_Paulo',
    }
    r_om = requests.get(
        'https://archive-api.open-meteo.com/v1/archive',
        params=params_om, timeout=60
    )
    r_om.raise_for_status()
    climate_raw = r_om.json()

    # Step 3: Summarize to 5-year station averages
    YEARS = 5
    station_climate = []
    for i, loc in enumerate(climate_raw):
        d = pd.DataFrame(loc['daily']).apply(pd.to_numeric, errors='coerce')
        station_climate.append({
            'station_code':       stations_sp.iloc[i]['CD_ESTACAO'],
            'station_name':       stations_sp.iloc[i]['DC_NOME'],
            'lat':                loc['latitude'],
            'lon':                loc['longitude'],
            'temp_mean_c':        round(d['temperature_2m_mean'].mean(), 1),
            'hot_days_yr':        round((d['temperature_2m_max'] > 35).sum() / YEARS, 1),
            'comfortable_days_yr': round(d['temperature_2m_mean'].between(18, 26).sum() / YEARS, 1),
            'precip_annual_mm':   round(d['precipitation_sum'].sum() / YEARS, 0),
            'humidity_mean_pct':  round(d['relative_humidity_2m_mean'].mean(), 1),
            'sunshine_hrs_yr':    round(d['sunshine_duration'].sum() / YEARS / 3600, 0),
        })
    climate_stations = pd.DataFrame(station_climate)
    print(f'\nClimate summary for {len(climate_stations)} stations (2019–2023 average):')
    display(climate_stations[[
        'station_name', 'temp_mean_c', 'hot_days_yr',
        'comfortable_days_yr', 'precip_annual_mm', 'sunshine_hrs_yr'
    ]])
else:
    climate_stations = None


In [ ]:
# Assign each municipality its nearest INMET station using Haversine distance,
# then inherit that station's climate profile.
# Municipality centroids are derived from the IBGE GeoJSON boundaries.
def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorised Haversine distance (km)."""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

if climate_stations is not None:
    import geopandas as gpd, io, warnings
    # Compute centroids from the IBGE GeoJSON (same source used for maps)
    geo_url = (
        'https://servicodados.ibge.gov.br/api/v3/malhas/estados/35'
        '?formato=application/vnd.geo+json&qualidade=minima&intrarregiao=municipio'
    )
    gdf_cent = gpd.read_file(io.BytesIO(requests.get(geo_url, timeout=30).content))
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        gdf_cent['lat'] = gdf_cent.geometry.centroid.y
        gdf_cent['lon'] = gdf_cent.geometry.centroid.x
    mun_coord = gdf_cent[['codarea', 'lat', 'lon']].rename(columns={'codarea': 'cod_ibge'})
    mun_coord['cod_ibge'] = mun_coord['cod_ibge'].astype(int)
    print(f'Municipality centroids: {len(mun_coord)}')
    # Nearest-station assignment
    st_lats = climate_stations['lat'].values
    st_lons = climate_stations['lon'].values
    nearest_idx = [
        int(np.argmin(haversine_km(row['lat'], row['lon'], st_lats, st_lons)))
        for _, row in mun_coord.iterrows()
    ]
    mun_coord = mun_coord.copy()
    mun_coord['nearest_station'] = climate_stations.iloc[nearest_idx]['station_code'].values
    CLIMATE_COLS = ['temp_mean_c', 'hot_days_yr', 'comfortable_days_yr',
                    'precip_annual_mm', 'humidity_mean_pct', 'sunshine_hrs_yr']
    mun_climate = mun_coord.merge(
        climate_stations[['station_code'] + CLIMATE_COLS],
        left_on='nearest_station', right_on='station_code'
    )
    df = df.merge(mun_climate[['cod_ibge'] + CLIMATE_COLS], on='cod_ibge', how='left')
    print(f'Climate merged — coverage: {df["temp_mean_c"].notna().sum()}/{len(df)} municipalities')
    print(df[CLIMATE_COLS].describe().round(1))
else:
    for col in ['temp_mean_c', 'hot_days_yr', 'comfortable_days_yr',
                'precip_annual_mm', 'humidity_mean_pct', 'sunshine_hrs_yr']:
        df[col] = float('nan')
    print('Climate data not available.')


### 7. Territorial Area & Population Density — IBGE SIDRA


In [ ]:
# Table 615 - Territorial area (km²) per municipality
url_area = (
    'https://apisidra.ibge.gov.br/values'
    '/t/615/n6/all/v/allxp/p/last'
)

try:
    resp_area = requests.get(url_area, timeout=30)
    area_raw = resp_area.json()
    area_df = pd.DataFrame(area_raw[1:])
    area_df['cod_ibge'] = area_df['D1C'].astype(int)
    area_df['area_km2'] = pd.to_numeric(area_df['V'], errors='coerce')
    
    area_sp = area_df[area_df['cod_ibge'].astype(str).str.startswith('35')][['cod_ibge', 'area_km2']].copy()
    
    df = df.merge(area_sp, on='cod_ibge', how='left')
    df['densidade_hab_km2'] = (df['populacao_2022'] / df['area_km2']).round(1)
    
    print(f'Area data merged for {area_sp["cod_ibge"].nunique()} municipalities')
    df.sort_values('densidade_hab_km2', ascending=False).head(10)[['nome', 'populacao_2022', 'area_km2', 'densidade_hab_km2']]
except Exception as e:
    print(f'Could not fetch area data: {e}')
    print('Alternative: IBGE provides area in the "Área Territorial" table')

---
## Composite Liveability Score

Each indicator is normalized to 0–1 with `MinMaxScaler`, then combined as a weighted sum.
Default weights are shown below — use the **Interactive Explorer** in the next section to override them live.


In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Define indicators and desired direction (higher=better or lower=better)
# All indicators below are fetched automatically — no manual download needed.
# forest_pct and temp_mean_c remain optional (require MapBiomas/INMET data).
scoring_config = {
    'pib_per_capita':    {'weight': 0.25, 'higher_better': True},  # GDP per capita (R$)
    'idhm':              {'weight': 0.30, 'higher_better': True},  # Human Development Index 2010
    'densidade_hab_km2': {'weight': 0.20, 'higher_better': False}, # pop density — prefer less dense
    'area_km2':          {'weight': 0.05, 'higher_better': True},  # territory size (tiebreaker)
    # Optional — uncomment when data is loaded:
    'forest_pct':          {'weight': 0.08, 'higher_better': True},  # % forest/natural vegetation
    'temp_mean_c':         {'weight': 0.05, 'higher_better': False}, # annual mean temp (prefer mild)
    'comfortable_days_yr': {'weight': 0.07, 'higher_better': True},  # days/yr with temp 18–26°C
    'hot_days_yr':         {'weight': 0.05, 'higher_better': False}, # days/yr above 35°C
    'sunshine_hrs_yr':     {'weight': 0.03, 'higher_better': True},  # annual sunshine hours
}

# Work with available columns only
available = {k: v for k, v in scoring_config.items() if k in df.columns and df[k].notna().sum() > 50}
print(f'Indicators available for scoring: {list(available.keys())}')
missing = [k for k in scoring_config if k not in available]
if missing:
    print(f'Skipped (not yet loaded): {missing}')

# Normalize each indicator to 0-1
scaler = MinMaxScaler()
df_scoring = df.dropna(subset=list(available.keys())).copy()
for col, config in available.items():
    normalized = scaler.fit_transform(df_scoring[[col]])
    if not config['higher_better']:
        normalized = 1 - normalized  # invert so higher = better
    df_scoring[f'{col}_norm'] = normalized

# Weighted score — weights re-normalised to sum to 1 over available indicators
total_weight = sum(v['weight'] for v in available.values())
df_scoring['livability_score'] = sum(
    df_scoring[f'{col}_norm'] * (config['weight'] / total_weight)
    for col, config in available.items()
)
df_scoring['rank'] = df_scoring['livability_score'].rank(ascending=False).astype(int)
print(f'\n\U0001f3c6 Top 25 municipalities by livability score ({len(df_scoring)} scored):')

top_cols = ['rank', 'nome', 'mesorregiao', 'populacao_2022'] + list(available.keys()) + ['livability_score']
df_scoring.sort_values('livability_score', ascending=False).head(25)[top_cols]


---
## 🎮 Interactive Liveability Explorer

Set a **1–5 priority** for each metric. The ranking updates instantly.
- **1** = ignore &nbsp; **3** = moderate &nbsp; **5** = critical
- Use the population filter to focus on a specific city-size band.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML
from sklearn.preprocessing import MinMaxScaler
import math

# ── Metric definitions (grouped by category) ─────────────────────────────────
METRIC_GROUPS = {
    'Quality of Life': {
        'idhm':              {'label': 'Human Development Index (IDHM)',      'higher_better': True,  'default': 5},
        'pib_per_capita':    {'label': 'GDP per Capita (R$)',                  'higher_better': True,  'default': 4},
        'densidade_hab_km2': {'label': 'Population Density (less = better)',   'higher_better': False, 'default': 3},
    },
    'Environment': {
        'forest_pct':        {'label': 'Forest / Green Coverage (%)',          'higher_better': True,  'default': 3},
        'area_km2':          {'label': 'Territory Area (km²)',                 'higher_better': True,  'default': 1},
    },
    'Climate': {
        'comfortable_days_yr': {'label': 'Comfortable Days/yr (18–26°C)',      'higher_better': True,  'default': 4},
        'hot_days_yr':         {'label': 'Hot Days/yr avoided (>35°C)',         'higher_better': False, 'default': 3},
        'temp_mean_c':         {'label': 'Mean Temperature (lower = milder)',   'higher_better': False, 'default': 2},
        'sunshine_hrs_yr':     {'label': 'Annual Sunshine Hours',               'higher_better': True,  'default': 2},
        'precip_annual_mm':    {'label': 'Annual Precipitation (mm)',           'higher_better': True,  'default': 1},
        'humidity_mean_pct':   {'label': 'Mean Humidity (%) — lower = drier',  'higher_better': False, 'default': 1},
    },
}

METRICS = {k: v for group in METRIC_GROUPS.values() for k, v in group.items()}

# Only keep metrics that actually have data
ACTIVE = {k: v for k, v in METRICS.items() if k in df.columns and df[k].notna().sum() > 50}

# ── Widgets ───────────────────────────────────────────────────────────────────
sliders = {
    col: widgets.IntSlider(
        value=cfg['default'], min=1, max=5, step=1,
        description='', continuous_update=True,
        style={'description_width': '0px'},
        layout=widgets.Layout(width='220px')
    )
    for col, cfg in ACTIVE.items()
}

pop_min_slider = widgets.IntSlider(
    value=20_000, min=0, max=500_000, step=5_000,
    description='', continuous_update=False,
    style={'description_width': '0px'},
    layout=widgets.Layout(width='300px')
)
pop_max_slider = widgets.IntSlider(
    value=600_000, min=5_000, max=12_000_000, step=10_000,
    description='', continuous_update=False,
    style={'description_width': '0px'},
    layout=widgets.Layout(width='300px')
)
top_n_slider = widgets.IntSlider(
    value=25, min=5, max=100, step=5,
    description='', continuous_update=False,
    style={'description_width': '0px'},
    layout=widgets.Layout(width='200px')
)

# ── Hard climate filters ──────────────────────────────────────────────────────
def _safe(val, default):
    return default if (val is None or (isinstance(val, float) and math.isnan(val))) else float(val)

_temp_max = _safe(df['temp_mean_c'].max()         if 'temp_mean_c'         in df.columns else None, 35.0)
_temp_min = _safe(df['temp_mean_c'].min()         if 'temp_mean_c'         in df.columns else None, 10.0)
_comf_max = _safe(df['comfortable_days_yr'].max() if 'comfortable_days_yr' in df.columns else None, 365.0)
_comf_min = _safe(df['comfortable_days_yr'].min() if 'comfortable_days_yr' in df.columns else None, 0.0)
_hot_max  = _safe(df['hot_days_yr'].max()         if 'hot_days_yr'         in df.columns else None, 100.0)

climate_temp_max_slider = widgets.FloatSlider(
    value=round(_temp_max, 1), min=round(_temp_min, 1), max=round(_temp_max, 1), step=0.5,
    description='', continuous_update=False,
    style={'description_width': '0px'},
    layout=widgets.Layout(width='300px'),
    readout_format='.1f',
)
climate_comf_min_slider = widgets.IntSlider(
    value=int(_comf_min), min=int(_comf_min), max=int(_comf_max), step=5,
    description='', continuous_update=False,
    style={'description_width': '0px'},
    layout=widgets.Layout(width='300px'),
)
climate_hot_max_slider = widgets.IntSlider(
    value=int(_hot_max), min=0, max=int(_hot_max), step=1,
    description='', continuous_update=False,
    style={'description_width': '0px'},
    layout=widgets.Layout(width='300px'),
)

out = widgets.Output()

# ── Recalculation ─────────────────────────────────────────────────────────────
def recalc(_=None):
    weights = {col: sliders[col].value for col in ACTIVE}
    total_w = sum(weights.values())
    if total_w == 0:
        return

    pop_lo = pop_min_slider.value
    pop_hi = pop_max_slider.value
    top_n  = top_n_slider.value

    subset = df[df['populacao_2022'].between(pop_lo, pop_hi)].copy()

    # Hard climate filters (only applied when the column has data)
    if 'temp_mean_c' in df.columns:
        subset = subset[subset['temp_mean_c'].isna() | (subset['temp_mean_c'] <= climate_temp_max_slider.value)]
    if 'comfortable_days_yr' in df.columns:
        subset = subset[subset['comfortable_days_yr'].isna() | (subset['comfortable_days_yr'] >= climate_comf_min_slider.value)]
    if 'hot_days_yr' in df.columns:
        subset = subset[subset['hot_days_yr'].isna() | (subset['hot_days_yr'] <= climate_hot_max_slider.value)]

    subset = subset.dropna(subset=list(ACTIVE.keys())).copy()

    if subset.empty:
        with out:
            out.clear_output(wait=True)
            print('No municipalities match the current filters.')
        return

    scaler = MinMaxScaler()
    for col, cfg in ACTIVE.items():
        norm = scaler.fit_transform(subset[[col]])
        if not cfg['higher_better']:
            norm = 1 - norm
        subset[f'_n_{col}'] = norm

    subset['livability_score'] = sum(
        subset[f'_n_{col}'] * (weights[col] / total_w)
        for col in ACTIVE
    )
    subset['rank'] = subset['livability_score'].rank(ascending=False).astype(int)
    top = subset.sort_values('rank').head(top_n)

    display_cols = ['rank', 'nome', 'mesorregiao', 'populacao_2022'] + list(ACTIVE.keys()) + ['livability_score']
    display_cols = [c for c in display_cols if c in top.columns]

    fmt = {
        'populacao_2022':      '{:,.0f}',
        'pib_per_capita':      'R$ {:,.0f}',
        'densidade_hab_km2':   '{:.1f}',
        'idhm':                '{:.3f}',
        'forest_pct':          '{:.1f}%',
        'temp_mean_c':         '{:.1f}°C',
        'comfortable_days_yr': '{:.0f} days',
        'hot_days_yr':         '{:.0f} days',
        'sunshine_hrs_yr':     '{:,.0f} hrs',
        'precip_annual_mm':    '{:,.0f} mm',
        'humidity_mean_pct':   '{:.1f}%',
        'area_km2':            '{:,.0f}',
        'livability_score':    '{:.4f}',
    }

    styled = (
        top[display_cols]
        .style
        .format({c: f for c, f in fmt.items() if c in display_cols})
        .bar(subset=['livability_score'], color='#4caf8a', vmin=0, vmax=1)
        .set_caption(
            f'Top {min(top_n, len(top))} of {len(subset):,} municipalities '
            f'(pop {pop_lo:,}-{pop_hi:,}) | {len(subset)} eligible'
        )
        .set_table_styles([{
            'selector': 'caption',
            'props': [('font-size', '13px'), ('font-weight', 'bold'), ('text-align', 'left')]
        }])
    )

    with out:
        out.clear_output(wait=True)
        display(styled)

# Attach observers
for sl in (
    list(sliders.values()) +
    [pop_min_slider, pop_max_slider, top_n_slider,
     climate_temp_max_slider, climate_comf_min_slider, climate_hot_max_slider]
):
    sl.observe(recalc, names='value')

# ── Layout helpers ─────────────────────────────────────────────────────────────
def labeled_row(label, wgt):
    lbl = widgets.HTML(
        f'<b style="font-size:13px">{label}</b>',
        layout=widgets.Layout(width='310px')
    )
    return widgets.HBox([lbl, wgt], layout=widgets.Layout(align_items='center', margin='2px 0'))

def section_header(title, color='#555'):
    html = (
        '<div style="margin:8px 0 4px; padding:3px 6px; background:' + color + '; '
        'color:white; border-radius:3px; font-size:12px; font-weight:bold">'
        + title + '</div>'
    )
    return widgets.HTML(html)

# ── Metric priority box (grouped by category) ─────────────────────────────────
GROUP_COLORS = {'Quality of Life': '#1976d2', 'Environment': '#388e3c', 'Climate': '#e64a19'}

metric_rows = []
for group_name, group_metrics in METRIC_GROUPS.items():
    active_in_group = [(col, cfg) for col, cfg in group_metrics.items() if col in ACTIVE]
    if not active_in_group:
        continue
    metric_rows.append(section_header(group_name, GROUP_COLORS.get(group_name, '#555')))
    for col, cfg in active_in_group:
        metric_rows.append(labeled_row(cfg['label'], sliders[col]))

metric_box = widgets.VBox(
    [widgets.HTML('<h4 style="margin:6px 0">Metric priorities (1 = ignore - 5 = critical)</h4>')] +
    metric_rows,
    layout=widgets.Layout(border='1px solid #ddd', padding='10px', margin='0 10px 0 0', width='580px')
)

# ── Filter box (population + hard climate cutoffs) ─────────────────────────────
climate_filter_rows = []
if 'temp_mean_c' in df.columns:
    climate_filter_rows.append(labeled_row('Max mean temp (degrees C)', climate_temp_max_slider))
if 'comfortable_days_yr' in df.columns:
    climate_filter_rows.append(labeled_row('Min comfortable days/yr', climate_comf_min_slider))
if 'hot_days_yr' in df.columns:
    climate_filter_rows.append(labeled_row('Max hot days/yr (>35 degrees C)', climate_hot_max_slider))

filter_children = [
    widgets.HTML('<h4 style="margin:6px 0">Filters</h4>'),
    labeled_row('Min population', pop_min_slider),
    labeled_row('Max population', pop_max_slider),
    labeled_row('Show top N',     top_n_slider),
]
if climate_filter_rows:
    filter_children.append(section_header('Hard Climate Cutoffs', '#e64a19'))
    filter_children.extend(climate_filter_rows)

filter_box = widgets.VBox(
    filter_children,
    layout=widgets.Layout(border='1px solid #ddd', padding='10px', width='560px')
)

controls = widgets.HBox([metric_box, filter_box])
display(controls, out)
recalc()  # initial render


---
## 🗺️ Liveability Map

Composite score mapped across all 645 municipalities. Green = higher liveability.


In [ ]:
import geopandas as gpd
import io

geo_url = (
    'https://servicodados.ibge.gov.br/api/v3/malhas/estados/35'
    '?formato=application/vnd.geo+json&qualidade=minima&intrarregiao=municipio'
)
try:
    geo_resp = requests.get(geo_url, timeout=30)
    geo_resp.raise_for_status()
    gdf = gpd.read_file(io.BytesIO(geo_resp.content))
    gdf['codarea'] = gdf['codarea'].astype(int)
    gdf_merged = gdf.merge(df_scoring, left_on='codarea', right_on='cod_ibge', how='left')

    fig, ax = plt.subplots(1, 1, figsize=(14, 11))
    gdf_merged.plot(
        column='livability_score', cmap='RdYlGn', legend=True, ax=ax,
        missing_kwds={'color': 'lightgray'},
        legend_kwds={'shrink': 0.6, 'label': 'Liveability Score (0–1)'}
    )
    ax.set_title('Composite Liveability Score — São Paulo State Municipalities', fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    print(f'{len(gdf_merged)} municipalities plotted')
except Exception as e:
    print(f'Map error: {e}')


---
## Top 30 Municipalities


In [ ]:
# Visualize top municipalities
top30 = df_scoring.sort_values('livability_score', ascending=False).head(30)

fig, ax = plt.subplots(figsize=(14, 10))
ax.barh(top30['nome'][::-1], top30['livability_score'][::-1], color='teal', edgecolor='white')
ax.set_xlabel('Livability Score (0–1)')
ax.set_title('Top 30 SP Municipalities — Composite Livability Score')
plt.tight_layout()
plt.show()

### Filter by Population Band

Adjust `pop_min` / `pop_max` to focus on the city size that fits your lifestyle.


In [ ]:
# Filter: municipalities with population between a comfortable range
# (small enough to be livable, big enough to have services)
pop_min = 30_000
pop_max = 500_000

filtered = df_scoring[
    (df_scoring['populacao_2022'] >= pop_min) &
    (df_scoring['populacao_2022'] <= pop_max)
].sort_values('livability_score', ascending=False)

print(f'Municipalities with population {pop_min:,}–{pop_max:,}: {len(filtered)}')
print(f'\n🏆 Top 20 in this range:')
filtered.head(20)[top_cols]

---
## 12. Summary & Next Steps

### Datasets successfully accessed via API:
- ✅ Municipality list (IBGE Localidades API)  
- ✅ Population 2022 Census (IBGE SIDRA, table 9514)  
- ✅ Municipal GDP & GDP per capita (IBGE SIDRA, table 5938)  
- ✅ Territorial area (IBGE SIDRA, table 615)  
- ✅ INMET weather station catalog  
- ✅ IBGE municipality boundaries (GeoJSON)  

### Datasets requiring manual download:
- 📥 **IDHM** (Atlas Brasil): http://www.interse.atlasbrasil.org.br/consulta/  
- 📥 **MapBiomas** land cover: https://data.mapbiomas.org/  
- 📥 **SEADE IMP** indicators: https://repositorio.seade.gov.br/  
- 📥 **INMET** historical weather data: https://portal.inmet.gov.br/dadoshistoricos  
- 📥 **SSP-SP** crime statistics: https://www.ssp.sp.gov.br/estatistica/  
- 📥 **SNIS** sanitation data: http://app4.cidades.gov.br/serieHistorica/  

### Recommended next steps:
1. **Download IDHM data** and merge — this is the single most impactful quality-of-life indicator
2. **Download MapBiomas** CSV to quantify green coverage per municipality
3. **Pull INMET station data** for the municipalities you're interested in
4. **Add crime data** from SSP-SP for safety analysis
5. **Refine the scoring model** with all indicators and personalized weights
6. **Explore specific candidate towns** in depth (cost of living, commute to SP capital, etc.)

In [ ]:
# Export livability scores and all metrics per municipality
export_cols = [
    'cod_ibge', 'nome', 'microrregiao', 'mesorregiao',
    'populacao_2022', 'area_km2', 'densidade_hab_km2',
    'pib_per_capita', 'pib_mil_reais',
    'idhm',
    'forest_pct',
    'temp_mean_c', 'precip_annual_mm', 'humidity_mean_pct',
    'livability_score', 'rank',
]

# Merge rank/score back onto full df so unscored municipalities still appear (with NaN score)
scored_cols = ['cod_ibge', 'livability_score', 'rank']
df_export = df.merge(df_scoring[scored_cols], on='cod_ibge', how='left')
df_export = df_export[[c for c in export_cols if c in df_export.columns]]
df_export = df_export.sort_values('rank')

out_path = 'sp_municipalities_livability.csv'
df_export.to_csv(out_path, index=False, float_format='%.4f')
print(f'Exported {len(df_export)} municipalities to {out_path}')
print(f'Columns: {list(df_export.columns)}')
df_export.head(10)
